# Proyecto de extracción de datos de PDF a data estructurada
## 1. Instalamos y cargamos la librerias necesarias

In [1]:
# ============================================================================
# BLOQUE 0: CARGAMOS LIBRERIAS Y CONFIGURAMOS RUTAS
# ============================================================================

#pip install pdfplumber pandas tabula-py openpyxl  PyMuPDF
import fitz  # PyMuPDF
import pandas as pd #Para manejar dataframes
import re           #Para manejo de expresiones regulares
import os           #Para manejo de archivos y directorios
import pdfplumber  #Para extraer texto y tablas de PDFs
import tabula  #Para extraer tablas de PDFs
from tabula import read_pdf
import json         #Para manejar archivos JSON
import requests      #Para hacer solicitudes HTTP
import PyPDF2
from pathlib import Path
import time
from IPython.display import display, Markdown

In [2]:
# ============================================================================
# BLOQUE 1: VERIFICAR CONEXIÓN CON OLLAMA
# ============================================================================

# EJECUTAR EN: Python 3.8+ (Recomiendo Anaconda o Python base)
OLLAMA_URL = "http://localhost:11434/api/generate"
#MODELO = "deepseek-r1:7b"
#MODELO = "mistral:latest"
MODELO = "llama3:latest"
#MODELO = "llama3.1:latest" 20Gb RAM recomendado

def verificar_ollama():
    """Verifica que Ollama esté corriendo y el modelo disponible""" # Docstring
    try:
        # Verificar servidor
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.status_code == 200:
            print("✅ Servidor Ollama OK")
            
            # Verificar modelo
            modelos = r.json().get('models', [])
            for m in modelos:
                if MODELO in m.get('name', ''):
                    print(f"✅ Modelo {MODELO} disponible")
                    return True
            
            print(f"❌ Modelo {MODELO} no encontrado en Ollama")
            return False
            
    except:
        print("❌ No se pudo conectar a Ollama")
        return False

verificar_ollama()

✅ Servidor Ollama OK
✅ Modelo llama3:latest disponible


True

In [3]:
# ============================================================================
# BLOQUE 2: CONFIGURACIÓN RUTAS DE ARCHIVOS
# ============================================================================

PDF_PATH = "C:\\Users\\jach_\\Downloads\\FOV_Exp_119-2025-DSEM-CMIN_CA_24-5-2025-103_y_CA_40-7-2025-103_R__20251126092237276.pdf"  # Ruta del PDF a procesar

In [4]:
# ============================================================================
# BLOQUE 3: FUNCIONES BÁSICAS PARA PDF
# ============================================================================

# Función para leer PDF y extraer texto por páginas
def leer_pdf(ruta, paginas=None):
    """Lee el PDF y devuelve texto por páginas""" # Docstring
    texto_completo = ""  # Variable para acumular todo el texto
    
    with open(ruta, 'rb') as f:
        pdf = PyPDF2.PdfReader(f)
        total_paginas = len(pdf.pages)
        print(f"📄 Procesando {total_paginas} páginas...")
        
        # Itera por todas las páginas
        for i in range(total_paginas):
            contenido = pdf.pages[i].extract_text() or ""
            texto_completo += contenido + "\n"  # Añade un salto de línea simple
            print(f"  ✅ Página {i+1}: {len(contenido)} caracteres")
    
    # Limpieza opcional: elimina saltos de línea excesivos
    texto_completo = texto_completo.strip()
    
    print(f"📊 Texto total extraído: {len(texto_completo)} caracteres")
    return texto_completo

# Función para consultar a Ollama
def consultar(pregunta, contexto=""):
    """Consulta simple al modelo desplegado""" # Docstring
    prompt = f"""Contexto: {contexto[:5000]}   

Pregunta: {pregunta}

Responde de manera clara y concisa:"""
    
    try:
        r = requests.post(OLLAMA_URL, json={
            "model": MODELO,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.1}
        }, timeout=60)
        
        return r.json().get('response', 'Error')
    except Exception as e:
        return f"Error: {e}"

In [5]:
# ============================================================================
# BLOQUE 4: CARGAR EL PDF
# ============================================================================

# Verificar que el archivo existe
if not Path(PDF_PATH).exists():
    print(f"❌ No se encontró el archivo: {PDF_PATH}")
else:
    # Cargar todas las páginas
    texto_pdf = leer_pdf(PDF_PATH)

print(texto_pdf)  # Mostrar los primeros 1000 caracteres para verificar

📄 Procesando 12 páginas...
  ✅ Página 1: 2154 caracteres
  ✅ Página 2: 4185 caracteres
  ✅ Página 3: 4610 caracteres
  ✅ Página 4: 3368 caracteres
  ✅ Página 5: 4125 caracteres
  ✅ Página 6: 3899 caracteres
  ✅ Página 7: 3143 caracteres
  ✅ Página 8: 2325 caracteres
  ✅ Página 9: 3214 caracteres
  ✅ Página 10: 3046 caracteres
  ✅ Página 11: 1926 caracteres
  ✅ Página 12: 1196 caracteres
📊 Texto total extraído: 37195 caracteres
Ministerio  
del Ambiente  Organismo de Evaluación y Fiscalización 
Ambiental - OEFA  Dirección de Supervisión 
Ambiental en Energía y Minas  
  
 “Decenio de la Igualdad de Oportunidades para Mujeres y Hombres”  
“Año de la recuperación y consolidación de la economía peruana”  
 
                                                                                                                                                                Pág. 1 de 12                                                                          
FICHA  DE OBLIGACIONES  VERIFICADAS  EN 

In [12]:
# Preguntamos al modelo sobre la información de la primer pagina del PDF I. INFORMACIÓN GENERAL:
# Incluyendo numero de expediente, administrado, ruc, unidad fiscalizable, estado, etaoa y CUC. Responde en formato JSON con claves: "numero_expediente", "administrado", "ruc", "unidad_fiscalizable", "estado", "etaoa", "cuc". Si no encuentras alguno de estos datos, deja el valor como null.

Prompt_info_general = (
    """ 
    Extrae del siguiente texto según estas reglas:

- numero_expediente: formato como "0119-2025-DSEM-CMIN". Lista si hay varios.
- administrado: nombre(s). Lista.
- ruc: formato "20466327612". Lista.
- unidad_fiscalizable: unidad responsable. Lista.
- estado: string (ej. "En trámite"). Si hay celdas combinadas, infiere.
- etapa: string (ej. "Evaluación"). Si hay celdas combinadas, infiere.
- cuc: formato "0024-5-2025-103". Lista.

Responde SOLO con un JSON válido con esas claves. Si no encuentras un campo, usa null. Para campos múltiples, usa lista aunque sea un elemento.

Ejemplo:
{
  "numero_expediente": ["0119-2025-DSEM-CMIN"],
  "administrado": ["CONSTRUCTORA LOS ANDES S.A.C."],
  "ruc": ["20466327612"],
  "unidad_fiscalizable": ["DSEM"],
  "estado": "En trámite",
  "etapa": "Evaluación de documentación",
  "cuc": ["0024-5-2025-103"]
}

Texto:
[PEGA AQUÍ EL TEXTO]
    """)

# buscar por paginas 1, 2 e integrar resultados en formato json, solo agregar nueva informacion encontrada en la pagina 2.
# Unir pagina 1 y pagina 2 del PDF para obtener la información de fuente de obligaciones fiscalizables, 
# Limitar la busqueda desde donde aparece la palabra "I. INFORMACIÓN GENERAL" hasta donde aparece "II. FUENTE DE OBLIGACIONES FISCALIZABLES" del PDF.


inicio = texto_pdf.find("I. INFORMACIÓN GENERAL")
fin = texto_pdf.find("II. FUENTE DE OBLIGACIONES FISCALIZABLES")
texto_filtrado = texto_pdf[inicio:fin] if inicio != -1 and fin != -1 else texto_pdf

# luego consultar al modelo con el prompt definido anteriormente. 
# Agregar los resultados en una lista de diccionarios, 
# donde cada diccionario representa una fuente de obligación fiscalizable encontrada. 
# Imprimir el resultado en formato JSON.

Informacion_general = consultar(Prompt_info_general, texto_filtrado)

# Impimir la información general del expediente
print("\n📊 Información General del Expediente:")
print(Informacion_general)


📊 Información General del Expediente:
{
  "numero_expediente": ["0119-2025-DSEM-CMIN"],
  "administrado": ["ARUNTANI S.A.C."],
  "ruc": ["20466327612"],
  "unidad_fiscalizable": ["ARASI"],
  "estado": "En Actividad",
  "etapa": "Cierre",
  "cuc": ["0024-5-2025-103", "0040-7-2025-103"]
}


In [ ]:
# Preguntamos al modelo sobre la información de la primer pagina del PDF II. FUENTE DE OBLIGACIONES FISCALIZABLES :
# Incluyendo numero de fuente, administrado, ruc, unidad fiscalizable, estado, etapa y CUC. Responde en formato JSON con claves: "numero_expediente", "administrado", "ruc", "unidad_fiscalizable", "estado", "etaoa", "cuc". Si no encuentras alguno de estos datos, deja el valor como null.

Prompt_fuente_obligaciones = (
    "Consultar la información de fuente de obligaciones fiscalizables, incluyendo: " \
    "numero de fuente, tipo,fuente, autoridad competente, documento de aprobacion, fecha de aprobacion, descripcion, " \
    "la informacion del nro de fuente debe ser <F.1, F.2, etc> y puede ser mas de uno. " \
    "en autoriad competente siempre hay <SENACE, MINEM, etc.> y puede ser mas de uno. " \
    "en fuente puede ser <ITS, Reglamento etc.>. " \
    "en descripcion siempre debe haber <texto >" \
    "Responde en formato JSON con claves: " \
    "'numero_fuente', 'tipo', 'fuente', 'autoridad_competente', 'documento_aprobacion', 'fecha_aprobacion', 'descripcion'. " \
    "Si no encuentras alguno de estos datos, deja el valor como null.")

# buscar por paginas 1, 2 e integrar resultados en formato json, solo agregar nueva informacion encontrada en la pagina 2.
# Unir pagina 1 y pagina 2 del PDF para obtener la información de fuente de obligaciones fiscalizables, 
# Limitar la busqueda desde donde aparece la palabra "II. FUENTE DE OBLIGACIONES FISCALIZABLES" hasta donde aparece "III. OBLIGACIONES FISCALIZABLES" del PDF.

inicio = texto_pdf.find("II. FUENTE DE OBLIGACIONES FISCALIZABLES")
fin = texto_pdf.find("III. OBLIGACIONES FISCALIZABLES")
texto_filtrado = texto_pdf[inicio:fin] if inicio != -1 and fin != -1 else texto_pdf

# luego consultar al modelo con el prompt definido anteriormente. 
# Agregar los resultados en una lista de diccionarios, 
# donde cada diccionario representa una fuente de obligación fiscalizable encontrada. 
# Imprimir el resultado en formato JSON.

Fuente_obligaciones = consultar(Prompt_fuente_obligaciones, texto_filtrado)

print("\n📊 Resultados de la fuente de obligaciones fiscalizables:")
print(Fuente_obligaciones)


In [41]:
# Extraer la sección III. OBLIGACIONES FISCALIZABLES y limitar la búsqueda a partir de esta sección hasta el final del PDF.
def extraer_desde_seccion_III(texto):
    match = re.search(r"III\.\s*OBLIGACIONES\s*FISCALIZABLES", texto, re.IGNORECASE)
    return texto[match.start():] if match else texto

# Divide el texto cada vez que aparece 3.1 al inicio de una fila. hasta que aparezca 3.2 y asi sucesivamente 3.n hasta el final del texto.
# No crear un bloque por las filas que comiencen con 3.1.1 por ejemplo solo considerar 3.x, donde x es un numero.
# El resultado debe ser una lista de bloques de texto, donde cada bloque corresponde a una sección 3.x y su contenido asociado, sin separar las subsecciones 3.x.x.

def dividir_por_secciones(texto):
    """
    Divide el texto cada vez que aparece F.x al inicio de una fila.
    """
    filas = re.split(r'(?=^F\.\d+\s)', texto, flags=re.MULTILINE)
    return [f.strip() for f in filas if re.search(r'^F\.\d+\s', f, re.MULTILINE)]


# Extraer campos de cada bloque que contenga simultáneamente un valor tipo F.x y un valor tipo O.x, 
# y que no sea un encabezado de sección (ejemplo: 3.1, 3.1.1) con filas. 
# cada fila con su respectiva referencia (F.x) y nro de obligación (O.x). en el formato de las columnas en orden exacto como se muestra arriba.
# Columnas exactas en orden:
#1) Seccion -> formato: 3.x texto: ejemplo: 3.1 Mantenimiento y monitoreo post cierre
#2) Subseccion -> formato: 3.x.x texto: ejemplo: 3.1.1 Pad de lixiviacion Jessica  
#3) Referencia (Nro Fuente) -> formato: F.x
#4) Nro Obligación -> formato: O.x
#5) Ubicación
#6) Descripción de la Obligación
#7) Verificación de la Obligación / Descripción de la Conducta Detectada (Análisis)
#8) Medios Probatorios
#9) Cumplimiento


def extraer_campos(bloque):
    seccion = re.search(r'^(3\.\d+)\s+(.*)', bloque, re.MULTILINE)
    subseccion = re.search(r'^(3\.\d+\.\d+)\s+(.*)', bloque, re.MULTILINE)
    referencia = re.search(r'F\.\d+', bloque)
    nro_obligacion = re.search(r'O\.\d+', bloque)
    cumplimiento = re.search(r'(No cumple|Cumple|Cumple parcialmente)', bloque, re.IGNORECASE)

    return {
        "seccion": seccion.group(0) if seccion else None,
        "subseccion": subseccion.group(0) if subseccion else None,
        "referencia": referencia.group() if referencia else None,
        "nro_obligacion": nro_obligacion.group() if nro_obligacion else None,
        "cumplimiento": cumplimiento.group() if cumplimiento else None,
        "texto_completo": bloque
    }


In [44]:
secciones_obligaciones = extraer_desde_seccion_III(texto_pdf)
bloques = dividir_por_secciones(secciones_obligaciones)
for i, bloque in enumerate(bloques):
    print(f"{i, bloque}")  # Mostrar los primeros 500 caracteres de cada bloque

# no se ven los bloques, revisar la función dividir_por_secciones y extraer_desde_seccion_III 
# para asegurar que estén extrayendo correctamente la sección III y dividiendo por secciones 3.x.

(0, 'F.1 O.1 Informe N° \n318-2014 -\nMEM -\nDGAAM/D\nNAM/DGA\nM/PC  “(…) \nVII ACTIVIDADES DE MANTENIMIENTO Y MONITOREO  \n \n7.1 Actividades de mantenimiento. - \nMantenimiento físico. - Los componentes que contaran con \nmantenimiento físico se presentan en el cuadro siguiente:  \n \nCuadro N° 12 Mantenimiento Físico  \nZona  Componente  Código  \nAndrés  Tajo Valle  MN-01 \nTajo Carlos  MN-02 \nJessica  Tajo Jessica  MN-03 \nAndrés  Pad de lixiviación Andrés  IP-01 \nJessica  Pad de lixiviación Jessica  IP-08 \nAndrés  Depósito  de desmonte N° 1  MR-03 \nDepósito  de desmonte N° 3  MR-04 \nJessica  Depósito de desmonte \nJessica  MR-06 \n \nEn caso de detectar daños, fallas, rupturas se procederá a la \ncomunicación inmediata para dar inicio a las actividades de \nlimpieza, restauración, o reinstalación. Se realizará el 2° y 4° \ntrimestre los dos (02) primeros años y el tercer trimestre los tres \n(03) años siguientes.  \n \nMantenimiento Geoquímico. - consiste en detectar daños, 

In [51]:
# extraer campos de los bloques de forma automatizada y mostrar resultados
for i, bloque in enumerate(bloques):
    campos = extraer_campos(bloque)
    print(campos)


{'seccion': None, 'subseccion': None, 'referencia': 'F.1', 'nro_obligacion': 'O.1', 'cumplimiento': 'No cumple', 'texto_completo': 'F.1 O.1 Informe N° \n318-2014 -\nMEM -\nDGAAM/D\nNAM/DGA\nM/PC  “(…) \nVII ACTIVIDADES DE MANTENIMIENTO Y MONITOREO  \n \n7.1 Actividades de mantenimiento. - \nMantenimiento físico. - Los componentes que contaran con \nmantenimiento físico se presentan en el cuadro siguiente:  \n \nCuadro N° 12 Mantenimiento Físico  \nZona  Componente  Código  \nAndrés  Tajo Valle  MN-01 \nTajo Carlos  MN-02 \nJessica  Tajo Jessica  MN-03 \nAndrés  Pad de lixiviación Andrés  IP-01 \nJessica  Pad de lixiviación Jessica  IP-08 \nAndrés  Depósito  de desmonte N° 1  MR-03 \nDepósito  de desmonte N° 3  MR-04 \nJessica  Depósito de desmonte \nJessica  MR-06 \n \nEn caso de detectar daños, fallas, rupturas se procederá a la \ncomunicación inmediata para dar inicio a las actividades de \nlimpieza, restauración, o reinstalación. Se realizará el 2° y 4° \ntrimestre los dos (02) prim

In [8]:
# Preguntamos al modelo sobre la información de la pagina del PDF III. OBLIGACIONES FISCALIZABLES :
# Identificar los campos referencia (nro fuente), nro obligacion, ubicacion, descipcion de la obligacion, verificacion de la obligacion/descipcion de la conducta detectada (analisis), medios probatorios, cumplimiento.
# debajo de las cabeceras identificar la seccion que empieza con , 3.1 
# debajo en caso haya subsecciones 3.1.1, 3.1.2, etc, identificar la informacion de cada subseccion y agregarla a la informacion de la obligacion fiscalizable correspondiente.

Prompt_obligaciones = """
El siguiente texto proviene de una TABLA con columnas fijas.

Columnas exactas en orden:

1) Referencia (Nro Fuente) -> formato: F.x
2) Nro Obligación -> formato: O.x
3) Ubicación
4) Descripción de la Obligación
5) Verificación de la Obligación / Descripción de la Conducta Detectada (Análisis)
6) Medios Probatorios
7) Cumplimiento

IMPORTANTE:


Las columnas anteriores son fijas y deben respetarse en el orden indicado.
debajo de lascolumnas fijas puede haber filas como secciones (3.1, 3.2, etc) y subsecciones (3.1.1, 3.1.2, etc.) 
despues de estas secciones y subsecciones pueden haber filas que contengan la información de las obligaciones fiscalizables,
cada fila con su respectiva referencia (F.x) y nro de obligación (O.x). en el formato de las columnas en orden exacto como se muestra arriba.

- No mezcles columnas o filas.
- No resumas.
- Copia el texto exactamente como aparece.
- Si una celda está vacía, usar null.
- Devuelve SOLO JSON válido.
- No agregues explicaciones.

Formato obligatorio:

[
  {
    "seccion": "3.x texto: ejemplo: 3.1 Mantenimiento y monitoreo post cierre",
    "subseccion": "3.x.x texto: ejemplo: 3.1.1 Pad de lixiviacion Jessica",
    "referencia": "F.x",
    "nro_obligacion": "O.x",
    "ubicacion": "<.texto..>",
    "descripcion_obligacion": "<.texto..>",
    "verificacion_obligacion": "<.texto..>",
    "medios_probatorios": "<.texto..>",
    "cumplimiento": ["No cumple", "cumple", "texto"]
  }
]

Texto:
"""

# Limitar la busqueda desde donde aparece la palabra "III. OBLIGACIONES FISCALIZABLES" hasta el final del PDF.

inicio = texto_pdf.find("III. OBLIGACIONES FISCALIZABLES")
fin = len(texto_pdf)
texto_filtrado = texto_pdf[inicio:fin] if inicio != -1 else texto_pdf

# luego consultar al modelo con el prompt definido anteriormente. 
# Agregar los resultados en una lista de diccionarios, 
# donde cada diccionario representa una obligación fiscalizable encontrada. 
# Imprimir el resultado en formato JSON.

#Recorrer y extraer todo el texto  solo se queda en la primera hojan del PDF,
# integrar toda la informacion de las obligaciones fiscalizables de todas las paginas del PDF,
# Limitar la busqueda desde donde aparece la palabra "III. OBLIGACIONES FISCALIZABLES" hasta el final del PDF.

Obligaciones_fiscalizables = consultar(Prompt_obligaciones, texto_filtrado)

print("\n📊 Resultados de las obligaciones fiscalizables:")
print(Obligaciones_fiscalizables)



📊 Resultados de las obligaciones fiscalizables:
[
  {
    "seccion": "3.1 Mantenimiento y monitoreo post cierre",
    "subseccion": "3.1.1 Pad de lixiviación Jessica",
    "referencia": "F.1 O.1",
    "nro_obligacion": "O.1",
    "ubicacion": "Jessica",
    "descripcion_obligacion": "Mantenimiento físico. - Los componentes que contaran con mantenimiento físico se presentan en el cuadro siguiente:",
    "verificacion_obligacion": "De la obligación fiscalizable citada, respecto a las actividades en el Pad de lixiviación Aruntani debió realizar las siguientes actividades de mantenimiento: detectar fallas, rupturas en el componente y en las coberturas implementadas;",
    "medios_probatorios": ["• Ver hecho N° 1 del informe de supervisión.",
                          "• Acta de supervisión del 20 al 25 de mayo de 2025.",
                          "• Anexo 8.5. Registro fotográfico y fílmicos obtenidos durante la acción de supervisión del 20 al 25 de mayo de 2025.",
                       

In [ ]:
texto_filtrado = extraer_desde_seccion_III(texto_pdf)
print(texto_filtrado) 

In [96]:
texto_filas = dividir_por_filas(texto_filtrado)
texto_filas

['F.1 O.1 Informe N° \n318-2014 -\nMEM -\nDGAAM/D\nNAM/DGA\nM/PC  “(…) \nVII ACTIVIDADES DE MANTENIMIENTO Y MONITOREO  \n \n7.1 Actividades de mantenimiento. - \nMantenimiento físico. - Los componentes que contaran con \nmantenimiento físico se presentan en el cuadro siguiente:  \n \nCuadro N° 12 Mantenimiento Físico  \nZona  Componente  Código  \nAndrés  Tajo Valle  MN-01 \nTajo Carlos  MN-02 \nJessica  Tajo Jessica  MN-03 \nAndrés  Pad de lixiviación Andrés  IP-01 \nJessica  Pad de lixiviación Jessica  IP-08 \nAndrés  Depósito  de desmonte N° 1  MR-03 \nDepósito  de desmonte N° 3  MR-04 \nJessica  Depósito de desmonte \nJessica  MR-06 \n \nEn caso de detectar daños, fallas, rupturas se procederá a la \ncomunicación inmediata para dar inicio a las actividades de \nlimpieza, restauración, o reinstalación. Se realizará el 2° y 4° \ntrimestre los dos (02) primeros años y el tercer trimestre los tres \n(03) años siguientes.  \n \nMantenimiento Geoquímico. - consiste en detectar daños, fal

In [97]:
texto_campos = [extraer_campos(fila) for fila in texto_filas]
texto_campos

[{'referencia': 'F.1',
  'nro_obligacion': 'O.1',
  'cumplimiento': 'No cumple',
  'texto_completo': 'F.1 O.1 Informe N° \n318-2014 -\nMEM -\nDGAAM/D\nNAM/DGA\nM/PC  “(…) \nVII ACTIVIDADES DE MANTENIMIENTO Y MONITOREO  \n \n7.1 Actividades de mantenimiento. - \nMantenimiento físico. - Los componentes que contaran con \nmantenimiento físico se presentan en el cuadro siguiente:  \n \nCuadro N° 12 Mantenimiento Físico  \nZona  Componente  Código  \nAndrés  Tajo Valle  MN-01 \nTajo Carlos  MN-02 \nJessica  Tajo Jessica  MN-03 \nAndrés  Pad de lixiviación Andrés  IP-01 \nJessica  Pad de lixiviación Jessica  IP-08 \nAndrés  Depósito  de desmonte N° 1  MR-03 \nDepósito  de desmonte N° 3  MR-04 \nJessica  Depósito de desmonte \nJessica  MR-06 \n \nEn caso de detectar daños, fallas, rupturas se procederá a la \ncomunicación inmediata para dar inicio a las actividades de \nlimpieza, restauración, o reinstalación. Se realizará el 2° y 4° \ntrimestre los dos (02) primeros años y el tercer trimestr

## 2. Funciónes 
### 2.1 Función para extraer información del archivo PDF

### 2.2 Prompt de consulta

### 2.2 Inicializar la estructura de BD
##### Crea un diccionario principal con todas las secciones que vamos a extraer. Algunas son diccionarios (como datos individuales) y otras son listas (como equipos, personal, etc.).

### 2.3 Datos del Administrado
##### Extrae datos del administrado, recorre cada linea de texto , si encuentra una clave (RUC) extrae los numeros. Guarda en el diccionario: {"RUC": "20100177421"}

### 2.4 Datos de la supervisión
##### Usa una expresión regular para encontrar el texto que está entre "4 Datos de la Supervisión" y "5 Equipo de Supervisión". Esto captura toda la sección de datos de supervisión.

### 2.5 Equipo de supervisión
##### Patrón: Busca el patrón "1 : Nombre Completo : 40499423" (\d): Captura un dígito (el número) :: Literalmente dos puntos con espacios (.+?): Captura el nombre (cualquier texto) :: Otros dos puntos

### 2.6 Area supervisada

### 2.7 Verificación de obligaciones

### 2.8 Requerimientos realizados

### 2.9 Comentarios de las partes

### 2.10 Guardar en excel

### 2.12 Ejecutamos en conjunto todo el proceso